# Tarang - Standalone DSP Validation (Updated)

**Purpose:** Prove the causal, stateful DSP pipeline behaves correctly and identically in one-shot or streamed chunks.

**No model in the loop.** No TensorFlow, no Keras.

**Updated with:** SPKI/NPKI warm-up initialization, NLMS implementation, failing-record diagnostics, coupling-interval analysis, window-center alignment check, hard verdict gates.


## Section 1 - Environment

In [1]:
import os, sys, json, time, hashlib, platform
from pathlib import Path
from datetime import datetime
import numpy as np
import scipy
import scipy.signal
from scipy.signal import butter, sosfilt, sosfilt_zi, resample_poly, freqz, sosfreqz, find_peaks
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

DSP_DIR = '.'
sys.path.insert(0, DSP_DIR)
import tarang_dsp_reference as dsp

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_dspval"
ROOT_OUT = Path("artifacts/dsp_validation") / RUN_ID
SUBDIRS = ["00_config", "01_unit_tests", "02_chunk_invariance", "02_chunk_invariance/figures",
           "03_filter_characterization", "03_filter_characterization/figures",
           "04_detector_validation", "04_detector_validation/figures",
           "05_normalization_validation", "06_window_alignment", "06_window_alignment/figures",
           "07_nlms_ablation", "08_report"]
for d in SUBDIRS:
    (ROOT_OUT / d).mkdir(parents=True, exist_ok=True)

config = dsp.DSPConfig()
INCAR_DIR = r'C:\MMD Public\Hackathons\Team Ocelleon\dataset\incartdb'  # *** CHANGE THIS to your INCART data path on Windows ***

env_info = {"python": sys.version, "numpy": np.__version__, "scipy": scipy.__version__,
            "matplotlib": matplotlib.__version__, "platform": platform.platform(),
            "seed": SEED, "run_id": RUN_ID}
with open(ROOT_OUT / "00_config" / "environment.json", "w", encoding='utf-8') as f:
    json.dump(env_info, f, indent=2)
with open(ROOT_OUT / "00_config" / "dsp_config.json", "w", encoding='utf-8') as f:
    json.dump({k: getattr(config, k) for k in ['target_fs','morphology_low_hz','morphology_high_hz',
             'morphology_order','notch_enabled','notch_hz','normalization_window_sec',
             'detector_low_hz','detector_high_hz','detector_mwi_sec','refractory_sec',
             'recenter_ms','pre_r','post_r','nlms_mode','nlms_order','nlms_mu','nlms_delta']}, f, indent=2)

print(f"Run ID: {RUN_ID}")
print(f"Output: {ROOT_OUT}")
print(f"INCAR_DIR: {INCAR_DIR}")
print(f"INCART records available: ", end="")
incart_recs = []
for i in range(1, 76):
    r = f'I{i:02d}'
    if all(os.path.exists(f'{INCAR_DIR}/{r}.{ext}') for ext in ['hea','dat','atr']):
        incart_recs.append(r)
print(f"{len(incart_recs)} of 75")


Run ID: 20260801_003842_dspval
Output: artifacts\dsp_validation\20260801_003842_dspval
INCAR_DIR: C:\MMD Public\Hackathons\Team Ocelleon\dataset\incartdb
INCART records available: 75 of 75


## Section 2 - Unit Tests (Fixed)

18 tests including: real NLMS zero-reference/bypass/bounded-weight, 80% QRS threshold, RR arithmetic, window indexing.


In [2]:
test_results = {}
def run_test(name, condition, details=""):
    status = "PASS" if condition else "FAIL"
    test_results[name] = {"status": status, "details": details}
    print(f"  [{status}] {name}")
    return condition

print("=== UNIT TESTS ===\n")

# 1. Resample identity
s = rng.standard_normal(500).astype(np.float32)
run_test("250 Hz identity resampling", len(dsp.resample_signal(s, 250, 250)) == len(s))

# 2. Annotation rescaling
old = np.array([100, 200, 300, 400, 500])
new = np.round(old * 250.0 / 257).astype(int)
run_test("annotation rescaling", np.array_equal(new, np.array([97, 195, 292, 389, 486])))

# 3. Filter impulse response
sos = dsp.sos_design_bandpass(fs=250, lo=0.5, hi=40, order=4)
imp = np.zeros(2000); imp[0] = 1.0
st = dsp.sos_zero_state(sos); out = []
for x in imp:
    y, st = dsp.sos_step(float(x), st); out.append(y)
out = np.array(out)
run_test("filter impulse response", np.max(np.abs(out[-100:])) < 0.05 * np.max(np.abs(out)))

# 4. Filter step response
st = dsp.sos_zero_state(sos); out = []
for x in np.ones(2000):
    y, st = dsp.sos_step(float(x), st); out.append(y)
run_test("filter step response", abs(np.mean(out[-200:])) < 0.01)

# 5-7. Chunk invariance (one-shot vs 1-sample, random, 256)
sig = rng.standard_normal(10000).astype(np.float64)
for _ in range(25): sig[200 + _*400:200 + _*400 + 8] += np.sin(np.pi*np.arange(8)/8) * 5
sa = dsp.StreamingTarangDSP(config); pa = sa.process_record(sig)
for name, chunks in [("1-sample", [1]*10000), ("random", rng.integers(1,100,200)), ("256-sample", [256]*40)]:
    sc = dsp.StreamingTarangDSP(config); sc.warm_up(sig); sc._warmed_up = True
    pc = []
    idx = 0
    for cs in chunks:
        pc.extend(sc.process_frame(sig[idx:idx+cs])); idx += cs
    run_test(f"chunk invariance ({name})",
             len(pa) == len(pc) and all(a.r_peak_index == b.r_peak_index for a, b in zip(pa, pc)))

# 8. Causality
sa = dsp.StreamingTarangDSP(config); pa = sa.process_record(sig)
sb = sig.copy(); sb[5000:] += 5.0
sb_stream = dsp.StreamingTarangDSP(config); pb = sb_stream.process_record(sb)
safe_a = [p for p in pa if p.r_peak_index + 65 < 4900]
safe_b = [p for p in pb if p.r_peak_index + 65 < 4900]
run_test("causality", len(safe_a) == len(safe_b) and all(a.r_peak_index == b.r_peak_index for a, b in zip(safe_a, safe_b)))

# 9. Normalization startup
st = dsp.rolling_norm_init(window=7500); v = 0
for x in rng.standard_normal(8000) * 0.5:
    y, st = dsp.rolling_norm_step(float(x), st)
    if not np.isfinite(y): v += 1
run_test("normalization startup (no NaN/Inf)", v == 0)

# 10. No NaN/Inf in pipeline
sg = dsp.StreamingTarangDSP(config); pg = sg.process_record(rng.standard_normal(5000).astype(np.float64))
run_test("no NaN/Inf in pipeline", all(np.all(np.isfinite(p.waveform)) for p in pg))

# 11. Detector on synthetic QRS (80% threshold)
sq = np.zeros(5000)
expected = []
for p in range(500, 4500, 400):
    for j in range(8): sq[p+j] = np.sin(np.pi*j/8) * 5.0
    expected.append(p + 4)
sh = dsp.StreamingTarangDSP(config); ph = sh.process_record(sq)
det = [p.r_peak_index for p in ph]
matched = sum(1 for ep in expected if any(abs(d - ep) < 50 for d in det))
run_test("detector on synthetic QRS (80% threshold)", matched >= int(0.8 * len(expected)),
         f"matched={matched}/{len(expected)}")

# 12. Refractory
dups = sum(1 for i in range(1, len(ph)) if ph[i].r_peak_index - ph[i-1].r_peak_index < 50)
run_test("refractory duplicate-rejection", dups == 0)

# 13. RR arithmetic
ps = np.array([0.4, 0.8, 1.2, 1.6, 2.0, 2.4])
rr = np.array([(ps[-1]-ps[-2])*1000, np.mean(np.diff(ps[-5:])*1000), np.std(np.diff(ps[-5:])*1000), 60000/np.mean(np.diff(ps[-5:])*1000)])
run_test("RR feature arithmetic", np.allclose(rr, [400, 400, 0, 150], atol=0.01))

# 14. Window indexing
run_test("130-sample window indexing", len(ph[len(ph)//2].waveform) == 130)

# 15. Annotation matching
r = dsp.match_detected_peaks_to_annotations([10,50,100,200], [12,55,105,300], 10)
run_test("annotation matching", r['tp']==3 and r['fp']==1 and r['fn']==1)

# 16. NLMS zero-reference (zero IMU -> zero correction)
ns = dsp.nlms_init(order=16, active=True)
y, ns2 = dsp.nlms_step(1.0, np.zeros(16), ns)
run_test("NLMS zero-reference (zero IMU -> zero correction)", abs(y - 1.0) < 1e-12 and np.all(ns2.weights == 0))

# 17. NLMS bypass (output = input in bypass mode)
ns_b = dsp.nlms_init(order=16, active=False)
y_b, _ = dsp.nlms_step(2.5, np.ones(16), ns_b)
run_test("NLMS bypass mode (output = input)", abs(y_b - 2.5) < 1e-12)

# 18. NLMS bounded-weight (weights stay finite)
ns_w = dsp.nlms_init(order=16, mu=0.01, active=True)
for _ in range(1000):
    y, ns_w = dsp.nlms_step(rng.standard_normal(), rng.standard_normal(16)*100, ns_w)
run_test("NLMS bounded-weight", np.isfinite(ns_w.weights).all() and np.linalg.norm(ns_w.weights) < 1e6)

with open(ROOT_OUT / "01_unit_tests" / "test_results.json", "w", encoding='utf-8') as f:
    json.dump({k: {kk: (bool(vv) if isinstance(vv, (np.bool_,)) else vv) for kk, vv in v.items()}
               for k, v in test_results.items()}, f, indent=2)

n_pass = sum(1 for v in test_results.values() if v["status"] == "PASS")
n_fail = sum(1 for v in test_results.values() if v["status"] == "FAIL")
print(f"\n=== SUMMARY: {n_pass} PASS, {n_fail} FAIL out of {n_pass+n_fail} ===")


=== UNIT TESTS ===

  [PASS] 250 Hz identity resampling
  [PASS] annotation rescaling
  [PASS] filter impulse response
  [PASS] filter step response
  [FAIL] chunk invariance (1-sample)
  [FAIL] chunk invariance (random)
  [FAIL] chunk invariance (256-sample)
  [PASS] causality
  [PASS] normalization startup (no NaN/Inf)
  [PASS] no NaN/Inf in pipeline
  [PASS] detector on synthetic QRS (80% threshold)
  [PASS] refractory duplicate-rejection
  [PASS] RR feature arithmetic
  [PASS] 130-sample window indexing
  [PASS] annotation matching
  [PASS] NLMS zero-reference (zero IMU -> zero correction)
  [PASS] NLMS bypass mode (output = input)
  [PASS] NLMS bounded-weight

=== SUMMARY: 15 PASS, 3 FAIL out of 18 ===


## Section 3 - Chunk Invariance and Causality

In [3]:
# Chunk invariance
print("=== CHUNK INVARIANCE ===\n")
sig_ci = rng.standard_normal(10000).astype(np.float64)
for _ in range(25): sig_ci[200+_*400:200+_*400+8] += np.sin(np.pi*np.arange(8)/8) * 5

ci_results = {}
for method in ['one_shot', '1_sample', '256_sample', 'random']:
    stream = dsp.StreamingTarangDSP(config)
    if method == 'one_shot':
        pkts = stream.process_record(sig_ci)
    else:
        stream.warm_up(sig_ci); stream._warmed_up = True
        pkts = []
        if method == '1_sample':
            for x in sig_ci: pkts.extend(stream.process_sample(float(x)))
        elif method == '256_sample':
            for i in range(0, len(sig_ci), 256): pkts.extend(stream.process_frame(sig_ci[i:i+256]))
        else:
            idx = 0
            for cs in rng.integers(1, 100, 200):
                pkts.extend(stream.process_frame(sig_ci[idx:idx+cs])); idx += cs
    ci_results[method] = [p.r_peak_index for p in pkts]

ref = ci_results['one_shot']
all_ci_pass = True
for method, peaks in ci_results.items():
    if method == 'one_shot': continue
    match = len(peaks) == len(ref) and all(a == b for a, b in zip(ref, peaks))
    if not match: all_ci_pass = False
    print(f"  {method:15s}: {len(peaks)} beats, match={match}")

fig, ax = plt.subplots(figsize=(14, 4), constrained_layout=True)
for i, (method, peaks) in enumerate(ci_results.items()):
    ax.scatter(peaks, [i]*len(peaks), label=f'{method} ({len(peaks)})', s=20, alpha=0.7)
ax.set_yticks(range(len(ci_results))); ax.set_yticklabels(ci_results.keys())
ax.set_title('Chunk Invariance: Beat R-peak Indices by Method')
ax.legend(); ax.grid(True, alpha=0.3)
fig.savefig(str(ROOT_OUT / "02_chunk_invariance" / "figures" / "chunk_invariance.png"), dpi=120)
plt.close()

# Causality
print("\n=== CAUSALITY ===\n")
sa = dsp.StreamingTarangDSP(config); pa = sa.process_record(sig_ci)
sb = sig_ci.copy(); sb[5000:] += 5.0
sb_s = dsp.StreamingTarangDSP(config); pb = sb_s.process_record(sb)
safe_a = [p for p in pa if p.r_peak_index + 65 < 4900]
safe_b = [p for p in pb if p.r_peak_index + 65 < 4900]
causal_pass = len(safe_a) == len(safe_b) and all(a.r_peak_index == b.r_peak_index for a, b in zip(safe_a, safe_b))
print(f"  Safe beats: {len(safe_a)} (A) vs {len(safe_b)} (B), pass={causal_pass}")

fig, ax = plt.subplots(figsize=(14, 4), constrained_layout=True)
ax.vlines([p.r_peak_index for p in pa], 0, 1, colors='blue', alpha=0.5, label=f'Unperturbed ({len(pa)})')
ax.vlines([p.r_peak_index for p in pb], 0, 1, colors='red', alpha=0.5, linestyle='--', label=f'Perturbed ({len(pb)})')
ax.axvline(x=5000, color='black', linestyle=':', linewidth=2, label='Perturb at 5000')
ax.set_title('Causality Test'); ax.legend(); ax.grid(True, alpha=0.3)
fig.savefig(str(ROOT_OUT / "02_chunk_invariance" / "figures" / "causality.png"), dpi=120)
plt.close()

print(f"\nVERDICT: Chunk={ 'PASS' if all_ci_pass else 'FAIL'}, Causality={'PASS' if causal_pass else 'FAIL'}")


=== CHUNK INVARIANCE ===

  1_sample       : 25 beats, match=False
  256_sample     : 25 beats, match=False
  random         : 25 beats, match=False

=== CAUSALITY ===

  Safe beats: 27 (A) vs 27 (B), pass=True

VERDICT: Chunk=FAIL, Causality=PASS


## Section 4 - Filter Characterization

In [4]:
print("=== FILTER CHARACTERIZATION ===\n")
sos_m = dsp.sos_design_bandpass(fs=250, lo=0.5, hi=40, order=4)
sos_h, sos_l = dsp.qrs_bandpass_design(fs=250, lo=5, hi=15, order=2)
sos_d = np.vstack([sos_h, sos_l])

def char_filter(sos, n=2000):
    imp = np.zeros(n); imp[0] = 1.0
    st = dsp.sos_zero_state(sos); ir = []
    for x in imp: y, st = dsp.sos_step(float(x), st); ir.append(y)
    st = dsp.sos_zero_state(sos); sr = []
    for x in np.ones(n): y, st = dsp.sos_step(float(x), st); sr.append(y)
    w, h = sosfreqz(sos, worN=4096, fs=250)
    return np.array(ir), np.array(sr), w, 20*np.log10(np.abs(h)+1e-12), np.unwrap(np.angle(h))

ir_m, sr_m, w_m, mag_m, ph_m = char_filter(sos_m)
ir_d, sr_d, w_d, mag_d, ph_d = char_filter(sos_d)

fig, axes = plt.subplots(3, 2, figsize=(16, 12), constrained_layout=True)
axes[0,0].plot(ir_m, 'b', lw=0.8); axes[0,0].set_title('Morphology Bandpass - Impulse')
axes[0,1].plot(sr_m, 'b', lw=0.8); axes[0,1].set_title('Morphology Bandpass - Step')
axes[1,0].plot(w_m, mag_m, 'b', lw=1); axes[1,0].set_title('Morphology - Frequency Response (dB)')
axes[1,0].set_xlim(0, 125); axes[1,0].set_ylim(-80, 5); axes[1,0].axvline(x=0.5, color='r', ls='--'); axes[1,0].axvline(x=40, color='r', ls='--')
axes[1,1].plot(w_d, mag_d, 'r', lw=1); axes[1,1].set_title('Detector - Frequency Response (dB)')
axes[1,1].set_xlim(0, 125); axes[1,1].set_ylim(-80, 5); axes[1,1].axvline(x=5, color='r', ls='--'); axes[1,1].axvline(x=15, color='r', ls='--')
axes[2,0].plot(ir_d, 'r', lw=0.8); axes[2,0].set_title('Detector - Impulse')
axes[2,1].plot(w_d, ph_d, 'r', lw=1); axes[2,1].set_title('Detector - Phase')
for ax in axes.flat: ax.grid(True, alpha=0.3)
fig.savefig(str(ROOT_OUT / "03_filter_characterization" / "figures" / "filter_char.png"), dpi=120)
plt.close()

filter_data = {'morph_peak': float(np.max(np.abs(ir_m))), 'morph_steady': float(np.mean(sr_m[-200:])),
               'det_peak': float(np.max(np.abs(ir_d))), 'det_steady': float(np.mean(sr_d[-200:]))}
with open(ROOT_OUT / "03_filter_characterization" / "impulse_response.json", "w", encoding='utf-8') as f:
    json.dump(filter_data, f, indent=2)
print(f"  Morphology: peak={filter_data['morph_peak']:.4e}, steady={filter_data['morph_steady']:.6e}")
print(f"  Detector:   peak={filter_data['det_peak']:.4e}, steady={filter_data['det_steady']:.6e}")
print("  Graph saved.")


=== FILTER CHARACTERIZATION ===

  Morphology: peak=3.4560e-01, steady=7.167411e-05
  Detector:   peak=1.2145e-01, steady=4.440892e-16
  Graph saved.


## Section 5 - Detector Validation (with SPKI/NPKI Warm-up)

Runs the Pan-Tompkins detector on all available INCART records. The warm-up fix initializes SPKI/NPKI from the first 2 seconds of signal, preventing the chicken-and-egg trap where TH1=0 causes noise-triggered false detections.

**If you see 0 records here, your INCART path is wrong.** Change `INCAR_DIR` in Section 1.


In [5]:
import wfdb
print("=== DETECTOR VALIDATION (with SPKI/NPKI warm-up) ===\n")

if len(incart_recs) == 0:
    print("WARNING: No INCART records found! Skipping detector validation.")
    print(f"Check that INCAR_DIR = '{INCAR_DIR}' has .hea/.dat/.atr files.")
    valid = {}; all_timing = np.array([]); agg_prec = agg_rec = agg_f1 = 0.0
else:
    all_results = {}
    for rec_name in incart_recs:
        try:
            rec = wfdb.rdrecord(f'{INCAR_DIR}/{rec_name}', channels=[0])
            ann = wfdb.rdann(f'{INCAR_DIR}/{rec_name}', 'atr')
            raw = rec.p_signal[:, 0].astype(np.float64)
            sig = dsp.resample_signal(raw, rec.fs, 250)
            ann_t = np.round(ann.sample.astype(np.float64) * 250 / rec.fs).astype(int)
            stream = dsp.StreamingTarangDSP(config)
            packets = stream.process_record(sig)
            fp = [p.r_peak_index for p in packets if p.quality_state == 'GOOD' or 'STARTUP' not in p.quality_flags]
            tp = [int(s) for s, sym in zip(ann_t, ann.symbol) if dsp.map_aami_symbol(sym) != 'IGNORE']
            tl = [dsp.map_aami_symbol(sym) for sym in ann.symbol if dsp.map_aami_symbol(sym) != 'IGNORE']
            mr = dsp.match_detected_peaks_to_annotations(fp, tp, 37)
            mt = dsp.evaluate_detector(mr)
            pc = {}
            for cls in ['N','S','V']:
                ci = [j for j, l in enumerate(tl) if l == cls]
                cm = sum(1 for j in ci if any(m[1] == j for m in mr['matches']))
                pc[cls] = {'total': len(ci), 'detected': cm, 'recall': cm/max(len(ci),1)}
            all_results[rec_name] = {'precision': mt['precision'], 'recall': mt['recall'], 'f1': mt['f1'],
                'tp': mr['tp'], 'fp': mr['fp'], 'fn': mr['fn'], 'per_class': pc,
                'timing_errors_ms': mt['timing_errors_ms'],
                'v_beats': [{'sample': int(tp[j]), 'matched': j in set(m[1] for m in mr['matches'])}
                            for j in range(len(tl)) if tl[j] == 'V'],
                'all_true_peaks': tp, 'all_true_labels': tl}
            print(f"  {rec_name}: prec={mt['precision']:.3f} rec={mt['recall']:.3f} "
                  f"N={pc['N']['recall']:.2f} V={pc['V']['recall']:.2f}")
        except Exception as e:
            all_results[rec_name] = {'error': str(e)[:100]}
            print(f"  {rec_name}: ERROR {type(e).__name__}")

    valid = {k: v for k, v in all_results.items() if 'error' not in v}
    ttp = sum(v['tp'] for v in valid.values())
    tfp = sum(v['fp'] for v in valid.values())
    tfn = sum(v['fn'] for v in valid.values())
    agg_prec = ttp / max(ttp+tfp, 1)
    agg_rec = ttp / max(ttp+tfn, 1)
    agg_f1 = 2*agg_prec*agg_rec / max(agg_prec+agg_rec, 1e-9)
    all_timing = np.array([t for v in valid.values() for t in v['timing_errors_ms']])

def safe_mean(a): return float(np.mean(a)) if len(a) > 0 else 0.0
def safe_std(a): return float(np.std(a)) if len(a) > 0 else 0.0
def safe_pct(a, q): return float(np.percentile(a, q)) if len(a) > 0 else 0.0

print(f"\n=== AGGREGATE ({len(valid)} records, of {len(incart_recs)} available, of 75 total) ===")
print(f"  Precision: {agg_prec:.4f}")
print(f"  Recall:    {agg_rec:.4f}")
print(f"  F1:        {agg_f1:.4f}")
if len(all_timing) > 0:
    print(f"  Timing:    mean={safe_mean(all_timing):+.2f}ms, std={safe_std(all_timing):.2f}ms")
for cls in ['N','S','V']:
    tot = sum(v['per_class'][cls]['total'] for v in valid.values()) if valid else 0
    det = sum(v['per_class'][cls]['detected'] for v in valid.values()) if valid else 0
    print(f"  {cls} recall: {det/max(tot,1):.4f} ({det}/{tot})")

# Save metrics
detector_json = {'n_records': len(valid), 'n_available': len(incart_recs), 'n_total': 75,
    'aggregate': {'precision': agg_prec, 'recall': agg_rec, 'f1': agg_f1,
                  'timing_mean_ms': safe_mean(all_timing), 'timing_std_ms': safe_std(all_timing)},
    'per_record': {k: {kk: vv for kk, vv in v.items() if kk not in ('timing_errors_ms','v_beats','all_true_peaks','all_true_labels')}
                   for k, v in valid.items()}}
with open(ROOT_OUT / "04_detector_validation" / "detector_metrics.json", "w", encoding='utf-8') as f:
    json.dump(detector_json, f, indent=2, default=str)

# Graphs
if len(valid) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
    rs = sorted(valid.keys())
    x = np.arange(len(rs))
    axes[0,0].bar(x-0.175, [valid[r]['precision'] for r in rs], 0.35, label='Precision', color='blue', alpha=0.7)
    axes[0,0].bar(x+0.175, [valid[r]['recall'] for r in rs], 0.35, label='Recall', color='orange', alpha=0.7)
    axes[0,0].set_title('Per-Record Precision and Recall')
    axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(rs, rotation=90, fontsize=6); axes[0,0].legend()
    vr = [valid[r]['per_class']['V']['recall'] for r in rs]
    axes[0,1].bar(x, vr, color=['red' if v<0.5 else 'green' for v in vr], alpha=0.7)
    axes[0,1].set_title('Per-Record V (PVC) Recall')
    axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(rs, rotation=90, fontsize=6)
    if len(all_timing) > 0:
        axes[1,0].hist(all_timing, bins=80, color='green', alpha=0.7, edgecolor='black')
        axes[1,0].axvline(x=safe_mean(all_timing), color='red', ls='--', label=f'Mean={safe_mean(all_timing):+.1f}ms')
        axes[1,0].set_title('Timing Error Distribution'); axes[1,0].legend()
    ar = [sum(v['per_class'][c]['detected'] for v in valid.values())/max(sum(v['per_class'][c]['total'] for v in valid.values()),1) for c in ['N','S','V']]
    axes[1,1].bar(['N','S','V'], ar, color=['blue','orange','red'], alpha=0.7)
    axes[1,1].set_title('Aggregate Recall by Class')
    for i, v in enumerate(ar): axes[1,1].text(i, v+0.02, f'{v:.3f}', ha='center')
    for ax in axes.flat: ax.grid(True, alpha=0.3)
    fig.savefig(str(ROOT_OUT / "04_detector_validation" / "figures" / "detector_validation.png"), dpi=120)
    plt.close()
    print(f"\n  Graph saved.")


=== DETECTOR VALIDATION (with SPKI/NPKI warm-up) ===

  I01: prec=1.000 rec=0.859 N=0.98 V=0.00
  I02: prec=1.000 rec=0.901 N=0.98 V=0.01
  I03: prec=1.000 rec=0.001 N=0.00 V=0.00
  I04: prec=0.471 rec=0.003 N=0.00 V=0.02
  I05: prec=0.988 rec=0.918 N=0.93 V=0.84
  I06: prec=1.000 rec=0.984 N=0.98 V=1.00
  I07: prec=1.000 rec=0.984 N=0.98 V=1.00
  I08: prec=0.710 rec=0.586 N=0.54 V=0.83
  I09: prec=0.997 rec=0.971 N=0.98 V=0.66
  I10: prec=1.000 rec=0.950 N=0.96 V=0.35
  I11: prec=0.988 rec=0.978 N=0.98 V=0.75
  I12: prec=0.717 rec=0.652 N=0.65 V=0.67
  I13: prec=0.667 rec=0.002 N=0.00 V=0.00
  I14: prec=0.500 rec=0.001 N=0.00 V=0.00
  I15: prec=0.971 rec=0.506 N=0.51 V=0.00
  I16: prec=0.980 rec=0.711 N=0.71 V=0.50
  I17: prec=0.858 rec=0.959 N=0.97 V=0.30
  I18: prec=0.704 rec=0.136 N=0.12 V=0.26
  I19: prec=0.992 rec=0.970 N=0.96 V=0.98
  I20: prec=0.839 rec=0.872 N=0.88 V=0.68
  I21: prec=0.811 rec=0.485 N=0.49 V=0.12
  I22: prec=0.926 rec=0.926 N=0.95 V=0.63
  I23: prec=1.000 rec=

## Section 6 - Coupling-Interval Analysis for Missed V Beats

Per review feedback: do NOT assume MWI merging is the cause. Measure it.

Three buckets:
- **< 200ms**: refractory window blocking (tunable parameter, NOT structural)
- **200-250ms**: MWI energy overlap (potential structural limit)
- **>= 250ms**: longer separation (investigate - could be T-wave rejection, SPKI calibration, or other)

Only computed on records with overall recall > 0.50 (excluding detector-broken records).

**V recall gate decision**: if >= 70% of misses fall under 250ms, treat as known limitation with soft floor. Otherwise, gate on it (it's a bug).


In [6]:
print("=== COUPLING-INTERVAL ANALYSIS ===\n")

if len(valid) == 0:
    print("No valid records - skipping coupling-interval analysis.")
    v_misses_ci = np.array([])
    pct_under_250 = 0
    v_gate_decision = "SKIP (no data)"
else:
    # Only use records with overall recall > 0.50
    good_records = {k: v for k, v in valid.items() if v['recall'] > 0.50}
    n_excluded = len(valid) - len(good_records)
    print(f"Records with recall > 0.50: {len(good_records)} (excluded {n_excluded} broken records)")

    # Collect missed V beats and their coupling intervals
    v_misses_ci = []
    v_hits_ci = []
    for rec_name, v in good_records.items():
        true_peaks = v.get('all_true_peaks', [])
        true_labels = v.get('all_true_labels', [])
        v_beats = v.get('v_beats', [])
        for vb in v_beats:
            v_sample = vb['sample']
            v_matched = vb['matched']
            # Find preceding true beat (any class)
            preceding = [tp for tp in true_peaks if tp < v_sample]
            if len(preceding) == 0:
                continue
            prev_beat = max(preceding)
            coupling_ms = (v_sample - prev_beat) * 1000.0 / 250
            if v_matched:
                v_hits_ci.append(coupling_ms)
            else:
                v_misses_ci.append(coupling_ms)

    v_misses_ci = np.array(v_misses_ci) if v_misses_ci else np.array([])
    v_hits_ci = np.array(v_hits_ci) if v_hits_ci else np.array([])

    # Three buckets
    n_miss = len(v_misses_ci)
    n_under_200 = sum(1 for c in v_misses_ci if c < 200) if n_miss > 0 else 0
    n_200_250 = sum(1 for c in v_misses_ci if 200 <= c < 250) if n_miss > 0 else 0
    n_over_250 = sum(1 for c in v_misses_ci if c >= 250) if n_miss > 0 else 0

    print(f"\nMissed V beats: {n_miss}")
    print(f"  < 200ms (refractory blocking):  {n_under_200} ({100*n_under_200/max(n_miss,1):.1f}%)")
    print(f"  200-250ms (MWI overlap):        {n_200_250} ({100*n_200_250/max(n_miss,1):.1f}%)")
    print(f"  >= 250ms (longer separation):   {n_over_250} ({100*n_over_250/max(n_miss,1):.1f}%)")
    print(f"\nDetected V beats: {len(v_hits_ci)}")
    if len(v_hits_ci) > 0:
        print(f"  Coupling interval: mean={np.mean(v_hits_ci):.1f}ms, min={np.min(v_hits_ci):.1f}ms, max={np.max(v_hits_ci):.1f}ms")

    # Gate decision: >= 70% under 250ms = known limitation
    pct_under_250 = (n_under_200 + n_200_250) / max(n_miss, 1) * 100
    if pct_under_250 >= 70:
        v_gate_decision = "KNOWN LIMITATION (>=70% misses under 250ms = MWI/refractory structural limit)"
    elif n_miss == 0:
        v_gate_decision = "PASS (no misses to analyze)"
    else:
        v_gate_decision = "BUG (misses spread across RR intervals - investigate)"
    print(f"\n  V gate decision: {v_gate_decision}")
    print(f"  (% misses under 250ms: {pct_under_250:.1f}%)")

# Save coupling-interval data
ci_data = {
    'n_missed_v': int(len(v_misses_ci)),
    'n_detected_v': int(len(v_hits_ci) if 'v_hits_ci' in dir() else 0),
    'bucket_under_200ms': int(sum(1 for c in v_misses_ci if c < 200)) if len(v_misses_ci) > 0 else 0,
    'bucket_200_250ms': int(sum(1 for c in v_misses_ci if 200 <= c < 250)) if len(v_misses_ci) > 0 else 0,
    'bucket_over_250ms': int(sum(1 for c in v_misses_ci if c >= 250)) if len(v_misses_ci) > 0 else 0,
    'pct_under_250ms': float(pct_under_250),
    'gate_decision': v_gate_decision,
    'n_records_excluded': int(n_excluded) if 'n_excluded' in dir() else 0,
}
with open(ROOT_OUT / "04_detector_validation" / "coupling_interval_analysis.json", "w", encoding='utf-8') as f:
    json.dump(ci_data, f, indent=2, default=str)

# Graph
if len(v_misses_ci) > 0 or len(v_hits_ci) if 'v_hits_ci' in dir() else 0 > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    if len(v_misses_ci) > 0:
        axes[0].hist(v_misses_ci, bins=50, color='red', alpha=0.7, edgecolor='black', label=f'Missed V ({len(v_misses_ci)})')
        axes[0].axvline(x=200, color='blue', ls='--', lw=2, label='Refractory (200ms)')
        axes[0].axvline(x=250, color='green', ls='--', lw=2, label='MWI overlap (250ms)')
        axes[0].set_title('Coupling Interval - Missed V Beats')
        axes[0].set_xlabel('Coupling interval (ms)'); axes[0].set_ylabel('Count')
        axes[0].legend()
    if len(v_hits_ci) if 'v_hits_ci' in dir() else 0 > 0:
        axes[1].hist(v_hits_ci, bins=50, color='green', alpha=0.7, edgecolor='black', label=f'Detected V ({len(v_hits_ci)})')
        axes[1].set_title('Coupling Interval - Detected V Beats')
        axes[1].set_xlabel('Coupling interval (ms)'); axes[1].legend()
    for ax in axes: ax.grid(True, alpha=0.3)
    fig.savefig(str(ROOT_OUT / "04_detector_validation" / "figures" / "coupling_interval.png"), dpi=120)
    plt.close()
    print("  Graph saved.")


=== COUPLING-INTERVAL ANALYSIS ===

Records with recall > 0.50: 52 (excluded 23 broken records)

Missed V beats: 5255
  < 200ms (refractory blocking):  0 (0.0%)
  200-250ms (MWI overlap):        0 (0.0%)
  >= 250ms (longer separation):   5255 (100.0%)

Detected V beats: 8044
  Coupling interval: mean=515.8ms, min=276.0ms, max=1904.0ms

  V gate decision: BUG (misses spread across RR intervals - investigate)
  (% misses under 250ms: 0.0%)
  Graph saved.


## Section 7 - Window-Center Alignment (XQRS vs Pan-Tompkins)

Compares peak positions from the v16 Pan-Tompkins detector against XQRS (the v15 detector). If the offset is > 5 samples (20ms) on average, the Gate/SV models may need retraining against the new detector's windows.


In [7]:
import wfdb.processing as wp
print("=== WINDOW-CENTER ALIGNMENT (XQRS vs PT) ===\n")

if len(incart_recs) == 0:
    print("No INCART records - skipping.")
    xqrs_pt_offsets = np.array([])
else:
    xqrs_pt_offsets = []
    n_align_recs = min(10, len(incart_recs))
    for rec_name in incart_recs[:n_align_recs]:
        try:
            rec = wfdb.rdrecord(f'{INCAR_DIR}/{rec_name}', channels=[0])
            raw = rec.p_signal[:, 0].astype(np.float64)
            sig = dsp.resample_signal(raw, rec.fs, 250)
            # XQRS (v15 detector)
            xqrs_peaks = wp.xqrs_detect(sig=sig, fs=250, verbose=False)
            # Pan-Tompkins (v16 detector)
            stream = dsp.StreamingTarangDSP(config)
            packets = stream.process_record(sig)
            pt_peaks = [p.r_peak_index for p in packets if p.quality_state == 'GOOD' or 'STARTUP' not in p.quality_flags]
            # Match XQRS to PT (nearest, 150ms tolerance)
            for xp in xqrs_peaks:
                if len(pt_peaks) > 0:
                    nearest = min(pt_peaks, key=lambda p: abs(p - xp))
                    offset = nearest - xp
                    if abs(offset) <= 37:  # 150ms
                        xqrs_pt_offsets.append(offset)
        except Exception as e:
            print(f"  {rec_name}: ERROR {type(e).__name__}")

    xqrs_pt_offsets = np.array(xqrs_pt_offsets) if xqrs_pt_offsets else np.array([])
    if len(xqrs_pt_offsets) > 0:
        mean_off = np.mean(xqrs_pt_offsets)
        std_off = np.std(xqrs_pt_offsets)
        print(f"Matched peaks: {len(xqrs_pt_offsets)}")
        print(f"  Mean offset: {mean_off:+.2f} samples ({mean_off*4:+.2f}ms)")
        print(f"  Std:         {std_off:.2f} samples ({std_off*4:.2f}ms)")
        print(f"  P5:          {np.percentile(xqrs_pt_offsets, 5):+.2f}")
        print(f"  P95:         {np.percentile(xqrs_pt_offsets, 95):+.2f}")
        retrain_risk = abs(mean_off) > 5
        print(f"\n  Retrain risk: {'YES' if retrain_risk else 'NO'} (mean offset {'>' if retrain_risk else '<='} 5 samples = 20ms)")
    else:
        print("No matched peaks for comparison.")

# Graph
if len(xqrs_pt_offsets) > 0:
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    ax.hist(xqrs_pt_offsets, bins=50, color='purple', alpha=0.7, edgecolor='black')
    ax.axvline(x=0, color='black', ls='-', lw=1)
    ax.axvline(x=np.mean(xqrs_pt_offsets), color='red', ls='--', lw=2, label=f'Mean={np.mean(xqrs_pt_offsets):+.1f}')
    ax.set_title('XQRS vs Pan-Tompkins Peak Offset (samples, 1 sample = 4ms)')
    ax.set_xlabel('Offset (PT - XQRS, samples)'); ax.set_ylabel('Count')
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.savefig(str(ROOT_OUT / "06_window_alignment" / "figures" / "xqrs_vs_pt_offset.png"), dpi=120)
    plt.close()
    print("  Graph saved.")

align_data = {'n_matched': int(len(xqrs_pt_offsets)),
              'mean_offset_samples': float(np.mean(xqrs_pt_offsets)) if len(xqrs_pt_offsets) > 0 else 0.0,
              'std_offset_samples': float(np.std(xqrs_pt_offsets)) if len(xqrs_pt_offsets) > 0 else 0.0,
              'retrain_risk': bool(abs(np.mean(xqrs_pt_offsets)) > 5) if len(xqrs_pt_offsets) > 0 else False}
with open(ROOT_OUT / "06_window_alignment" / "window_alignment.json", "w", encoding='utf-8') as f:
    json.dump(align_data, f, indent=2, default=str)


=== WINDOW-CENTER ALIGNMENT (XQRS vs PT) ===

Matched peaks: 19466
  Mean offset: +1.90 samples (+7.60ms)
  Std:         3.59 samples (14.35ms)
  P5:          +1.00
  P95:         +3.00

  Retrain risk: NO (mean offset <= 5 samples = 20ms)
  Graph saved.


## Section 8 - Normalization and NLMS

In [8]:
print("=== NORMALIZATION ===\n")
norm_results = {}
sig_n = rng.standard_normal(5000) * 0.5
s1 = dsp.rolling_norm_init(7500); s2 = dsp.rolling_norm_init(7500)
o1 = []; o2 = []
for i in range(3000):
    y1, s1 = dsp.rolling_norm_step(float(sig_n[i]), s1); o1.append(y1)
    y2, s2 = dsp.rolling_norm_step(float(sig_n[i]), s2); o2.append(y2)
norm_results['no_future_leakage'] = bool(np.max(np.abs(np.array(o1) - np.array(o2))) < 1e-12)
s3 = dsp.rolling_norm_init(7500); s4 = dsp.rolling_norm_init(7505)
o3 = []; o4 = []
for i in range(5000):
    y3, s3 = dsp.rolling_norm_step(float(sig_n[i]), s3); o3.append(y3)
for i in range(0, 5000, 256):
    for x in sig_n[i:i+256]:
        y4, s4 = dsp.rolling_norm_step(float(x), s4); o4.append(y4)
norm_results['no_frame_reset'] = bool(np.max(np.abs(np.array(o3) - np.array(o4))) < 1e-12)
s5 = dsp.rolling_norm_init(7500)
y0, _ = dsp.rolling_norm_step(1.0, s5)
norm_results['finite_at_startup'] = bool(np.isfinite(y0))
s6 = dsp.rolling_norm_init(7500); counts = []
for _ in range(100):
    _, s6 = dsp.rolling_norm_step(float(rng.standard_normal()), s6); counts.append(s6.C)
norm_results['valid_count_ramp'] = bool(np.array_equal(counts, list(range(1, 101))))

for k, v in norm_results.items():
    print(f"  {k}: {'PASS' if v else 'FAIL'}")

with open(ROOT_OUT / "05_normalization_validation" / "normalization_checks.json", "w", encoding='utf-8') as f:
    json.dump({k: bool(v) for k, v in norm_results.items()}, f, indent=2)

# NLMS ablation
print("\n=== NLMS ABLATION ===\n")
nlms_result = {"status": "skipped", "reason": "no synchronized IMU data available", "nlms_mode": config.nlms_mode}
with open(ROOT_OUT / "07_nlms_ablation" / "nlms_ablation.json", "w", encoding='utf-8') as f:
    json.dump(nlms_result, f, indent=2)
print("  SKIPPED - no synchronized IMU data")


=== NORMALIZATION ===

  no_future_leakage: PASS
  no_frame_reset: PASS
  finite_at_startup: PASS
  valid_count_ramp: PASS

=== NLMS ABLATION ===

  SKIPPED - no synchronized IMU data


## Section 9 - Final Report

Hard verdict gates:
- N recall >= 0.90
- No individual record with overall recall < 0.50
- V recall: measured via coupling-interval analysis (Section 6)
- V recall soft floor: written to JSON for future regression detection
- Record count stated explicitly (23 of 75 in this environment)


In [9]:
print("=== GENERATING DSP_VALIDATION_REPORT.md ===\n")

unit_pass = sum(1 for v in test_results.values() if v["status"] == "PASS")
unit_fail = sum(1 for v in test_results.values() if v["status"] == "FAIL")

# Hard gates
n_recall = sum(v['per_class']['N']['detected'] for v in valid.values()) / max(sum(v['per_class']['N']['total'] for v in valid.values()), 1) if valid else 0
records_below_50 = [k for k, v in valid.items() if v['recall'] < 0.50] if valid else []
n_gate_pass = n_recall >= 0.90
no_record_below_50 = len(records_below_50) == 0
chunk_pass = all_ci_pass if 'all_ci_pass' in dir() else True
causal_pass = causal_pass if 'causal_pass' in dir() else True
norm_pass = all(norm_results.values())

# V recall soft floor
v_recall_agg = sum(v['per_class']['V']['detected'] for v in valid.values()) / max(sum(v['per_class']['V']['total'] for v in valid.values()), 1) if valid else 0
v_baseline = {"v_recall_baseline": float(v_recall_agg), "dsp_config": str(config),
              "n_records": len(valid), "run_id": RUN_ID}
with open(ROOT_OUT / "04_detector_validation" / "v_recall_baseline.json", "w", encoding='utf-8') as f:
    json.dump(v_baseline, f, indent=2, default=str)

# Window alignment
align_risk = align_data.get('retrain_risk', False) if 'align_data' in dir() else False

# Verdict
all_hard_gates = (unit_fail == 0 and n_gate_pass and no_record_below_50 and
                  chunk_pass and causal_pass and norm_pass)

lines = []
lines.append("# Tarang - Standalone DSP Validation Report")
lines.append("")
lines.append(f"**Run ID:** {RUN_ID}")
lines.append(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append(f"**Artifacts:** {ROOT_OUT}")
lines.append(f"**INCART records:** {len(valid)} processed of {len(incart_recs)} available of 75 total")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Validation Summary")
lines.append("")
verdict = "READY FOR FIRMWARE PORT" if all_hard_gates else "NOT READY - see failures below"
lines.append(f"**Verdict: {verdict}**")
lines.append("")
lines.append("| Gate | Status | Details |")
lines.append("|---|---|---|")
lines.append(f"| Unit tests | {'PASS' if unit_fail == 0 else 'FAIL'} | {unit_pass}/{unit_pass+unit_fail} passed |")
lines.append(f"| Chunk invariance | {'PASS' if chunk_pass else 'FAIL'} | All chunking methods identical |")
lines.append(f"| Causality | {'PASS' if causal_pass else 'FAIL'} | No future-sample leakage |")
lines.append(f"| Filter characterization | PASS | Impulse, step, freq response saved |")
lines.append(f"| Detector N recall | {'PASS' if n_gate_pass else 'FAIL'} | N recall = {n_recall:.4f} (target >= 0.90) |")
lines.append(f"| No record < 0.50 recall | {'PASS' if no_record_below_50 else 'FAIL'} | {len(records_below_50)} records below 0.50 |")
lines.append(f"| V recall (measured) | {v_gate_decision} | {v_recall_agg:.4f} aggregate, {ci_data.get('n_missed_v', 0)} missed |")
lines.append(f"| Window alignment | {'WARNING' if align_risk else 'PASS'} | Mean offset {align_data.get('mean_offset_samples', 0):+.2f} samples |")
lines.append(f"| Normalization | {'PASS' if norm_pass else 'FAIL'} | No future leakage, no frame reset |")
lines.append(f"| NLMS ablation | SKIPPED | No synchronized IMU data |")
lines.append("")
lines.append("---")
lines.append("")
lines.append("## Detector Validation")
lines.append("")
lines.append(f"**{len(valid)} of {len(incart_recs)} available records processed (of 75 total in INCART).**")
if len(valid) < 75:
    lines.append(f"Full re-run required on your machine with all 75 records.")
lines.append("")
lines.append(f"| Metric | Value |")
lines.append(f"|---|---|")
lines.append(f"| Precision | {agg_prec:.4f} |")
lines.append(f"| Recall | {agg_rec:.4f} |")
lines.append(f"| N recall | {n_recall:.4f} |")
lines.append(f"| V recall | {v_recall_agg:.4f} |")
lines.append(f"| Timing mean | {safe_mean(all_timing):+.2f} ms |")
lines.append("")
lines.append("## Coupling-Interval Analysis (V recall)")
lines.append("")
lines.append(f"| Bucket | Count | % |")
lines.append(f"|---|---|---|")
lines.append(f"| < 200ms (refractory) | {ci_data.get('bucket_under_200ms', 0)} | {100*ci_data.get('bucket_under_200ms', 0)/max(ci_data.get('n_missed_v',1),1):.1f}% |")
lines.append(f"| 200-250ms (MWI overlap) | {ci_data.get('bucket_200_250ms', 0)} | {100*ci_data.get('bucket_200_250ms', 0)/max(ci_data.get('n_missed_v',1),1):.1f}% |")
lines.append(f"| >= 250ms (longer) | {ci_data.get('bucket_over_250ms', 0)} | {100*ci_data.get('bucket_over_250ms', 0)/max(ci_data.get('n_missed_v',1),1):.1f}% |")
lines.append(f"| **Decision** | {v_gate_decision} |")
lines.append("")
if align_risk:
    lines.append("## Window Alignment WARNING")
    lines.append("")
    lines.append(f"Mean offset between XQRS and Pan-Tompkins peaks: {align_data.get('mean_offset_samples', 0):+.2f} samples ({align_data.get('mean_offset_samples', 0)*4:+.2f}ms)")
    lines.append("This exceeds 5 samples (20ms) - the Gate/SV models may need retraining against the new detector's windows.")
    lines.append("")
lines.append("---")
lines.append("")
lines.append("## Conclusion")
lines.append("")
if all_hard_gates and not align_risk:
    lines.append("The DSP pipeline is **ready to port to firmware**. All hard gates pass.")
elif all_hard_gates and align_risk:
    lines.append("The DSP pipeline passes all DSP validation gates, BUT window-center alignment shows a retrain risk. Address the offset before firmware integration.")
else:
    lines.append("The DSP pipeline has **failures that must be fixed**. See the table above.")
    if not n_gate_pass:
        lines.append(f"- N recall ({n_recall:.4f}) below 0.90 target")
    if not no_record_below_50:
        lines.append(f"- {len(records_below_50)} records with recall < 0.50: {records_below_50}")
lines.append("")
lines.append(f"**Validated on {len(valid)} of 75 INCART records. Full re-run required on your machine.**")
lines.append("")
lines.append("---")
lines.append(f"\n**Artifacts:** `{ROOT_OUT}`")

report_text = "\n".join(lines)
with open(ROOT_OUT / "08_report" / "DSP_VALIDATION_REPORT.md", "w", encoding='utf-8') as f:
    f.write(report_text)

print(report_text)
print(f"\n{'='*70}")
print(f"Report saved: {ROOT_OUT / '08_report' / 'DSP_VALIDATION_REPORT.md'}")


=== GENERATING DSP_VALIDATION_REPORT.md ===

# Tarang - Standalone DSP Validation Report

**Run ID:** 20260801_003842_dspval
**Generated:** 2026-08-01 01:11:49
**Artifacts:** artifacts\dsp_validation\20260801_003842_dspval
**INCART records:** 75 processed of 75 available of 75 total

---

## Validation Summary

**Verdict: NOT READY - see failures below**

| Gate | Status | Details |
|---|---|---|
| Unit tests | FAIL | 15/18 passed |
| Chunk invariance | FAIL | All chunking methods identical |
| Causality | PASS | No future-sample leakage |
| Filter characterization | PASS | Impulse, step, freq response saved |
| Detector N recall | FAIL | N recall = 0.6768 (target >= 0.90) |
| No record < 0.50 recall | FAIL | 23 records below 0.50 |
| V recall (measured) | BUG (misses spread across RR intervals - investigate) | 0.4434 aggregate, 5255 missed |
| Window alignment | PASS | Mean offset +1.90 samples |
| Normalization | PASS | No future leakage, no frame reset |
| NLMS ablation | SKIPPED | 